
# Project Report: Classification of "Wigglies" Data

**Team:** Sara Prasla (10001841), Richard Van Winkle (3214183)

**Date:** July 9, 2025

### 1. Introduction

The goal of this project is to develop a high-performing deep learning model to classify the "wigglies" data and compete in the associated Kaggle competition. The core task involves implementing, training, evaluating, and comparing at least two distinct neural network architectures: a Multi-Layer Perceptron (MLP) and a Convolutional Neural Network (CNN)..

We were provided with three datasets:
1.  `trainset.pth`: An annotated dataset used for training and validating our models.
2.  `testset_noLabels.pth`: An unannotated dataset containing the "wigglies" data that our final model must classify for the competition submission.

The primary challenge is to determine the most effective architecture for this specific dataset and to optimize its performance to achieve the highest possible accuracy. Our strategy focuses on a systematic comparison between a baseline MLP, which treats the data as a simple vector, and a more advanced CNN, which is designed to capture potential sequential or spatial patterns within the data. This report details our step-by-step methodology for model development, the training and evaluation process, and the justification for our final model choice.

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import pandas as pd
import os

#### 2. Data Preparation

Our first step is to load the data from the `.pth` files. We created two utility functions:
* `get_dataloaders`: Loads the `trainset.pth` file, which contains both data and labels. It then splits this data into a training set (80%) and a validation set (20%). The validation set is crucial for monitoring model performance on unseen data to prevent overfitting.
* `get_test_loader`: Loads the `testset_noLabels.pth` file for generating the final predictions for the Kaggle submission.

The `weights_only=False` argument is necessary because our data files contain `TensorDataset` objects, not just model weights.

In [14]:
def get_dataloaders(train_path='../data/trainset.pth', batch_size=64, valid_split=0.2):
    """Loads and splits the training data into training and validation dataloaders."""
    
    # weights_only=False is required for loading TensorDataset objects from your files
    train_data = torch.load(train_path, weights_only=False)
    
    total_size = len(train_data)
    val_size = int(total_size * valid_split)
    train_size = total_size - val_size

    train_ds, val_ds = random_split(train_data, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

def get_test_loader(test_path='../data/testset_noLabels.pth', batch_size=64):
    """Loads the test dataset and returns a DataLoader."""
    test_ds = torch.load(test_path, weights_only=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return test_loader

##### Model 2: Convolutional Neural Network (CNN)

While the MLP is a good baseline, it cannot capture sequential patterns in the data. Since the data is named "wigglies," we hypothesized that the order of the 40 features is important. A 1D CNN is designed specifically to find such patterns.

**Architecture:**
* We developed a `ResNetCNN`, which uses **residual (or skip) connections**. This advanced architecture allows us to train a deeper model more effectively by improving gradient flow and preventing performance degradation.
* The model consists of two `ResidualBlock` modules. Each block contains `Conv1d` layers to find patterns, `BatchNorm1d` to stabilize training, and `MaxPool1d` to make the learned features more robust.

# --- 1. Define the ResNetCNN Architecture ---

In [1]:
class PowerfulCNN(nn.Module):
    def __init__(self, input_dim=40, num_classes=10):
        super().__init__()
        
        # Reshape input from [batch_size, 40] to [batch_size, 1, 40]
        self.unflatten = nn.Unflatten(1, (1, input_dim))
        
        # Block 1: Find simple, local patterns
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), # Stabilizes training
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2) # Downsample
        )
        
        # Block 2: Combine patterns into more complex ones
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        
        # Classifier Head
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), # Reduces each channel to a single value
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.unflatten(x)
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.head(x)
        return x

NameError: name 'nn' is not defined

**Training:**
* The training process is similar to the MLP but uses a `CosineAnnealingLR` scheduler. This scheduler smoothly decreases the learning rate over time, which can be more effective at finding a good final solution.


# Configuration for multiple runs (for ensembling)

In [2]:
# --- Configuration ---
save_path = "../outputs/best_cnn.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters ---
lr = 1e-3
epochs = 200 # Can be high due to early stopping
batch_size = 64
weight_decay = 1e-4
early_stopping_patience = 20 # Stop after 20 epochs of no improvement

# --- Setup ---
model = PowerfulCNN(input_dim=40, num_classes=10).to(device)
train_loader, val_loader = get_dataloaders(batch_size=batch_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

# --- Training Loop ---
epochs_no_improve = 0
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
        _, pred = out.max(1)
        train_correct += pred.eq(yb).sum().item()
        train_total += yb.size(0)
    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            _, pred = out.max(1)
            val_correct += pred.eq(yb).sum().item()
            val_total += yb.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), save_path)
        print(f"-> New best model saved with Val Acc: {best_val_acc:.4f}")
    else:
        epochs_no_improve += 1

    if epochs_no_improve == early_stopping_patience:
        print(f"\nEarly stopping triggered after {early_stopping_patience} epochs with no improvement.")
        break

NameError: name 'torch' is not defined

In [ ]:
# --- Configuration ---
weights_path = "../outputs/best_cnn.pth"
output_csv = "../outputs/submission_cnn.csv"

# --- Load Model ---
# Re-initialize the model structure
model = PowerfulCNN(input_dim=40, num_classes=10)
# Load the saved weights
model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device)
model.eval()

# --- Load Test Data ---
test_loader = get_test_loader()

# --- Generate Predictions ---
print("\nGenerating predictions on the test set...")
all_preds = []
with torch.no_grad():
    for xb in test_loader:
        # The dataloader gives a list [data_batch], so we take the first element
        xb = xb[0].to(device)
        outputs = model(xb)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# --- Create Submission File ---
assert len(all_preds) == 1000, f"Error: Expected 1000 predictions, but got {len(all_preds)}"
df = pd.DataFrame({"ID": range(1000), "Category": all_preds})
df.to_csv(output_csv, index=False)

print(f"\nSuccessfully saved predictions to {output_csv}")
print("Submission file head:")
print(df.head())


Generating predictions on the test set...

Successfully saved predictions to ../outputs/submission_cnn.csv
Submission file head:
   ID  Category
0   0         1
1   1         8
2   2         1
3   3         9
4   4         9


##### Model 2: Convolutional Neural Network (CNN)

While the MLP is a good baseline, it cannot capture sequential patterns in the data. Since the data is named "wigglies," we hypothesized that the order of the 40 features is important. A 1D CNN is designed specifically to find such patterns.

**Architecture:**
* We developed a `ResNetCNN`, which uses **residual (or skip) connections**. This advanced architecture allows us to train a deeper model more effectively by improving gradient flow and preventing performance degradation.
* The model consists of two `ResidualBlock` modules. Each block contains `Conv1d` layers to find patterns, `BatchNorm1d` to stabilize training, and `MaxPool1d` to make the learned features more robust.

# --- 1. Define the ResNetCNN Architecture ---

In [18]:
# --- Cell 3: The ResNetCNN Model Definition ---

class ResidualBlock(nn.Module):
    """A residual block with a skip connection."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        # Shortcut connection to match dimensions if they differ
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual # The "skip connection" adds the original input
        out = self.relu(out)
        return out

class ResNetCNN(nn.Module):
    """A powerful CNN using residual blocks."""
    def __init__(self, input_dim=40, num_classes=10):
        super().__init__()
        self.unflatten = nn.Unflatten(1, (1, input_dim))
        self.block1 = ResidualBlock(in_channels=1, out_channels=32)
        self.pool1 = nn.MaxPool1d(2)
        self.block2 = ResidualBlock(in_channels=32, out_channels=64)
        self.pool2 = nn.MaxPool1d(2)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.unflatten(x)
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.head(x)
        return x

**Training:**
* The training process is similar to the MLP but uses a `CosineAnnealingLR` scheduler. This scheduler smoothly decreases the learning rate over time, which can be more effective at finding a good final solution.


# Configuration for multiple runs (for ensembling)

In [23]:
# --- Cell 4: Training Script (with ResNetCNN and Cosine Annealing) ---

# --- Configuration ---
# IMPORTANT: For ensembling, change this name each time you run this cell.
# For example: "best_resnet_1.pth", "best_resnet_2.pth", etc.
save_path = "../outputs/best_resnet_5.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters ---
lr = 1e-3
epochs = 80
batch_size = 64
weight_decay = 1e-4
early_stopping_patience = 30

# --- Setup ---
# CHANGE 1: Initialize the new ResNetCNN model
model = ResNetCNN(input_dim=40, num_classes=10).to(device)
train_loader, val_loader = get_dataloaders(batch_size=batch_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

# --- Training Loop ---
epochs_no_improve = 0
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
        _, pred = out.max(1)
        train_correct += pred.eq(yb).sum().item()
        train_total += yb.size(0)
    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            _, pred = out.max(1)
            val_correct += pred.eq(yb).sum().item()
            val_total += yb.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total

    
    # CHANGE 3: Update the scheduler step. It no longer needs val_loss.
    scheduler.step()

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), save_path)
        print(f"-> New best model saved with Val Acc: {best_val_acc:.4f}")
    else:
        epochs_no_improve += 1

    if epochs_no_improve == early_stopping_patience:
        print(f"\nEarly stopping triggered after {early_stopping_patience} epochs with no improvement.")
        break

Using device: cuda
Epoch 1/80 | Train Loss: 1.6466 Acc: 0.5409 | Val Loss: 1.3336 Acc: 0.6987
-> New best model saved with Val Acc: 0.6987
Epoch 2/80 | Train Loss: 0.9990 Acc: 0.8494 | Val Loss: 0.8880 Acc: 0.8638
-> New best model saved with Val Acc: 0.8638
Epoch 3/80 | Train Loss: 0.6097 Acc: 0.9294 | Val Loss: 0.5956 Acc: 0.9113
-> New best model saved with Val Acc: 0.9113
Epoch 4/80 | Train Loss: 0.3801 Acc: 0.9556 | Val Loss: 0.4075 Acc: 0.9275
-> New best model saved with Val Acc: 0.9275
Epoch 5/80 | Train Loss: 0.2605 Acc: 0.9672 | Val Loss: 0.3034 Acc: 0.9450
-> New best model saved with Val Acc: 0.9450
Epoch 6/80 | Train Loss: 0.1912 Acc: 0.9738 | Val Loss: 0.2645 Acc: 0.9437
Epoch 7/80 | Train Loss: 0.1448 Acc: 0.9816 | Val Loss: 0.2161 Acc: 0.9500
-> New best model saved with Val Acc: 0.9500
Epoch 8/80 | Train Loss: 0.1184 Acc: 0.9841 | Val Loss: 0.2155 Acc: 0.9413
Epoch 9/80 | Train Loss: 0.0980 Acc: 0.9881 | Val Loss: 0.1491 Acc: 0.9712
-> New best model saved with Val Acc

#### 4. Prediction and Final Submission

To achieve the best possible score, we use **ensembling**. Instead of relying on a single model, we trained our best architecture (the `ResNetCNN`) multiple times. Each training run produces a slightly different model due to random initialization. By loading all these models and averaging their predictions, we can create a more robust and accurate final submission that is less prone to the errors of any single model.RESNETT

In [24]:
# --- Cell 5: Ensembling Prediction Script ---

# IMPORTANT: You must redefine the model architecture here so PyTorch can load the weights.
# This makes the cell self-contained.
class ResidualBlock(nn.Module):
    # --- Paste the full ResidualBlock class from Cell 3 here ---
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv1d(in_channels, out_channels, kernel_size=1), nn.BatchNorm1d(out_channels))
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = self.relu(out)
        return out

class ResNetCNN(nn.Module):
    # --- Paste the full ResNetCNN class from Cell 3 here ---
    def __init__(self, input_dim=40, num_classes=10):
        super().__init__()
        self.unflatten = nn.Unflatten(1, (1, input_dim))
        self.block1 = ResidualBlock(in_channels=1, out_channels=32)
        self.pool1 = nn.MaxPool1d(2)
        self.block2 = ResidualBlock(in_channels=32, out_channels=64)
        self.pool2 = nn.MaxPool1d(2)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(64, num_classes))
    def forward(self, x):
        x = self.unflatten(x)
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.head(x)
        return x

# --- Configuration ---
# 1. List the paths to all the models you trained
model_paths = [
    "../outputs/best_resnet_1.pth",
    "../outputs/best_resnet_2.pth",
    "../outputs/best_resnet_3.pth",
    "../outputs/best_resnet_4.pth",
    "../outputs/best_resnet_5.pth"
]
output_csv = "../outputs/submission_ensemble_resnet.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load all models
models = []
for path in model_paths:
    model = ResNetCNN()
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    models.append(model)

# 3. Generate predictions by ensembling
test_loader = get_test_loader()
all_preds = []
with torch.no_grad():
    for xb_list in test_loader:
        xb = xb_list[0].to(device)
        all_logits = [model(xb) for model in models]
        avg_logits = torch.stack(all_logits).mean(dim=0)
        preds = avg_logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# 4. Save submission file
df = pd.DataFrame({"ID": range(len(all_preds)), "Category": all_preds})
df.to_csv(output_csv, index=False)
print(f"Successfully saved ensemble predictions to {output_csv}")

Successfully saved ensemble predictions to ../outputs/submission_ensemble_resnet.csv
